# RoPE: final-token probe loss across 12 layers

Probe next-token-trained checkpoints **t=4096 (Stage 1a)** and **t=1048576 (Stage 3)** in `Auto_rope_checkpoints`. Compare against original last-token-trained checkpoints **t=512 (Stage 1a)** and **t=2097152 (Stage 3)**. Fit mixed-tree linear probes and evaluate on the same fixed ternary-tree test split as the original notebook (32,768 examples per split, seeds 42/43, rules seed 12345678, C=0.1). Embedding plus layers 1–12 are measured.

Results and figures have dedicated `rope` filenames. Run cells in order from the repository root; the final plotting cell also runs independently once results exist. Cached probe results are checked against checkpoint SHA-256 and experiment metadata.

The final plot compares both training methods at Stage 1a and Stage 3 on relative layer k/D (D=12). Training steps and positional encodings differ, so these are stage comparisons rather than a controlled estimate of the training objective alone. Losses are normalized by log(8), matching the original saved results.


## 1. Identical seeded probe datasets


In [ ]:
import gc, sys, pickle, warnings
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

WORKSPACE = Path.cwd()
if not (WORKSPACE / 'models').exists(): raise RuntimeError('Run from the repository root.')
sys.path[:0] = [str(WORKSPACE), str(WORKSPACE / 'datasets' / 'rhm')]
import models
from mixed_random_hierarchy_model import (
    sample_padded_rules, create_probabilities,
    sample_data_from_labels_varying_tree_tensorized,
)

V, L, BIN = 8, 3, 2
M_2, M_3, D_MAX, PAD_TOKEN = BIN, BIN * V, 3 ** L, V
T = D_MAX - 1
MODEL_DEPTH = 12
STEPS = [4096, 1048576]
SEED_RULES, TRAIN_SEED, TEST_SEED = 12345678, 42, 43
N_TRAIN, N_TEST = 32768, 32768
NUM_HEADS, BATCH_SIZE, PROBE_C = 16, 256, 0.1
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
CHECKPOINTS = {step: WORKSPACE / 'Auto_rope_checkpoints' / f'run_L3_v8_bin2_ter16_P16777216_lr0.001_l20.03_bs128_depth12_demb_256_nheads_16_arr0_t{step}.pt' for step in STEPS}
RESULT_PATHS = {step: WORKSPACE / f'probe_results_step_{step}_mixed_train_ternary_test_last_token_depth12_rope.pkl' for step in STEPS}
RECOMPUTE = False  # Dedicated RoPE cache; never reuse the original model results.
torch.set_num_threads(min(4, torch.get_num_threads()))
# Cached stages need no checkpoint file.
missing = [CHECKPOINTS[step] for step in STEPS
           if (RECOMPUTE or not RESULT_PATHS[step].is_file())
           and not CHECKPOINTS[step].is_file()]
if missing: raise FileNotFoundError('Missing checkpoints:\n' + '\n'.join(map(str, missing)))

rules = sample_padded_rules(V, V, M_2, M_3, 2, 3, L, SEED_RULES)
mixed_probs = create_probabilities(M_2, M_3, L)
ternary_probs = {l: torch.cat([torch.zeros(M_2), torch.full((M_3,), 1/M_3)]) for l in range(L)}

def generate_sentences(n, probabilities, seed):
    state = torch.random.get_rng_state(); torch.manual_seed(seed)
    labels = torch.randint(0, V, (n,))
    sentences, _ = sample_data_from_labels_varying_tree_tensorized(labels, rules, probabilities, V, D_MAX)
    torch.random.set_rng_state(state)
    return sentences

def make_last_token_task(sentences):
    # Mirrors init.MySentenceDataLoader exactly.
    n = len(sentences); lengths = (sentences != PAD_TOKEN).sum(1)
    targets = sentences[torch.arange(n), lengths - 1]
    input_lengths = lengths - 1
    columns = torch.arange(T).unsqueeze(0).expand(n, T)
    source_columns = columns - T + input_lengths.unsqueeze(1)
    gathered = torch.gather(sentences[:, :T], 1, source_columns.clamp(min=0))
    inputs = torch.where(source_columns >= 0, gathered, torch.full_like(gathered, PAD_TOKEN))
    return inputs, targets, lengths

mixed_sentences = generate_sentences(N_TRAIN, mixed_probs, TRAIN_SEED)
ternary_sentences = generate_sentences(N_TEST, ternary_probs, TEST_SEED)
X_train_tokens, y_train, mixed_lengths = make_last_token_task(mixed_sentences)
X_test_tokens, y_test, ternary_lengths = make_last_token_task(ternary_sentences)
assert torch.all(ternary_lengths == D_MAX)
assert torch.any(mixed_lengths < D_MAX) and torch.any(mixed_lengths == D_MAX)
assert torch.all(y_train < V) and torch.all(y_test < V)
print(f'device={DEVICE}; mixed train={N_TRAIN:,}; ternary test={N_TEST:,}')
print(f'mixed lengths={mixed_lengths.min().item()}..{mixed_lengths.max().item()}; ternary lengths={ternary_lengths.unique().tolist()}')

## 2. RoPE model loading and probe functions


In [ ]:
def infer_arch(state_dict):
    width = state_dict['token_embedding_table.weight'].shape[1]
    n_layers = max(int(k.split('.')[1]) for k in state_dict if k.startswith('blocks.')) + 1
    return dict(vocab_size=state_dict['token_embedding_table.weight'].shape[0],
                block_size=state_dict['blocks.0.attn.tril'].shape[0],
                embedding_dim=width, num_heads=NUM_HEADS,
                ffwd_size=state_dict['blocks.0.ffwd.net.0.weight'].shape[0] // width,
                num_layers=n_layers, dropout=0.0,
                rope='position_embedding_table.weight' not in state_dict, share_emb=False)

def load_model(path):
    saved = torch.load(path, map_location='cpu', weights_only=False)
    # Compiled checkpoints prefix parameter names with _orig_mod.
    state_dict = {key.removeprefix('_orig_mod.'): value
                  for key, value in saved['model'].items()}
    arch = infer_arch(state_dict); assert arch['block_size'] == T
    if (arch['num_layers'], arch['embedding_dim']) != (MODEL_DEPTH, 256):
        raise ValueError(f'Expected 12 layers and embedding width 256, got {arch}')
    if not arch['rope']: raise ValueError(f'Expected a RoPE checkpoint: {path}')
    model = models.CLM(**arch); model.load_state_dict(state_dict)
    return model.to(DEVICE).eval(), arch, int(saved.get('step', path.stem.rsplit('_t', 1)[-1]))

@torch.no_grad()
def extract_last_position_activations(model, inputs):
    # Shape: [embedding + blocks, examples, hidden width]. No other token positions are retained.
    by_depth = [[] for _ in range(model.num_layers + 1)]
    for start in range(0, len(inputs), BATCH_SIZE):
        if start % (BATCH_SIZE * 16) == 0: print(f'  activations {start:,}/{len(inputs):,}', flush=True)
        batch = inputs[start:start + BATCH_SIZE].to(DEVICE); captured = {}
        hooks = [model.blocks[0].register_forward_pre_hook(
            lambda module, args: captured.update({0: args[0][:, -1].detach().cpu()}))]
        for depth, block in enumerate(model.blocks, 1):
            hooks.append(block.register_forward_hook(
                lambda module, args, output, depth=depth: captured.update({depth: output[:, -1].detach().cpu()})))
        model(batch)
        for hook in hooks: hook.remove()
        for depth in range(model.num_layers + 1): by_depth[depth].append(captured[depth].numpy())
    return np.stack([np.concatenate(chunks) for chunks in by_depth])

def probe_across_depth(train_acts, test_acts):
    results = {'train_accuracy': [], 'test_accuracy': [],
               'train_cross_entropy_normalized': [], 'test_cross_entropy_normalized': []}
    train_labels, test_labels = y_train.numpy(), y_test.numpy()
    for depth, (X_train, X_test) in enumerate(zip(train_acts, test_acts)):
        probe = Pipeline([('scaler', StandardScaler()), ('classifier', LogisticRegression(
            C=PROBE_C, solver='lbfgs', max_iter=5000, random_state=0))])
        probe.fit(X_train, train_labels)
        results['train_accuracy'].append(probe.score(X_train, train_labels))
        results['test_accuracy'].append(probe.score(X_test, test_labels))
        train_ce = log_loss(train_labels, probe.predict_proba(X_train), labels=np.arange(V)) / np.log(V)
        test_ce = log_loss(test_labels, probe.predict_proba(X_test), labels=np.arange(V)) / np.log(V)
        results['train_cross_entropy_normalized'].append(train_ce)
        results['test_cross_entropy_normalized'].append(test_ce)
        print(f"  depth {depth}: acc train={results['train_accuracy'][-1]:.4f}, test={results['test_accuracy'][-1]:.4f}; "
              f"CE/log(V) train={train_ce:.4f}, test={test_ce:.4f}", flush=True)
    counts = np.bincount(train_labels, minlength=V)
    results.update(chance_uniform=1/V, train_majority_baseline=counts.max()/counts.sum())
    return results

## 3. Fit missing next-token checkpoints, or reuse their dedicated caches

Only t=4096 needs new probes if t=1048576 has already been run.


In [ ]:
import hashlib
def checkpoint_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
for requested_step in STEPS:
    checkpoint_path = CHECKPOINTS[requested_step]
    result_path = RESULT_PATHS[requested_step]
    CACHE_METADATA = {
        'experiment': 'mixed_train_ternary_test_last_token_depth12_rope_v1',
        'checkpoint_step': requested_step,
        'checkpoint_file': checkpoint_path.name,
        'checkpoint_relative_path': str(checkpoint_path.relative_to(WORKSPACE)),
        'checkpoint_sha256': checkpoint_sha256(checkpoint_path),
        'probe_C': PROBE_C, 'n_train_mixed': N_TRAIN, 'n_test_ternary': N_TEST,
        'train_seed': TRAIN_SEED, 'test_seed': TEST_SEED, 'rules_seed': SEED_RULES,
        'probe_position': 'last input position', 'loss_normalization': 'log(8)',
    }
    cache_valid = False
    if result_path.is_file() and not RECOMPUTE:
        with result_path.open('rb') as handle:
            cached = pickle.load(handle)
        cache_valid = (all(cached.get(k) == v for k, v in CACHE_METADATA.items())
                       and all(len(cached.get('token_probe', {}).get(metric, [])) == 13
                               for metric in ('test_accuracy', 'test_cross_entropy_normalized')))
    if cache_valid:
        print('Reusing:', result_path, flush=True)
    else:
        model, arch, loaded_step = load_model(checkpoint_path)
        if loaded_step != requested_step:
            raise ValueError(f'Expected step {requested_step}, found {loaded_step}')
        print('RoPE checkpoint loaded:', arch, flush=True)
        print('Extracting mixed-train activations', flush=True)
        train_acts = extract_last_position_activations(model, X_train_tokens)
        print('Extracting ternary-test activations', flush=True)
        test_acts = extract_last_position_activations(model, X_test_tokens)
        result = probe_across_depth(train_acts, test_acts)
        payload = {**CACHE_METADATA, 'architecture': arch, 'token_probe': result}
        temporary_path = result_path.with_suffix('.pkl.tmp')
        with temporary_path.open('wb') as handle:
            pickle.dump(payload, handle)
        temporary_path.replace(result_path)
        print('Saved:', result_path, flush=True)
        del model, train_acts, test_acts
        gc.collect()


## 4. Last-token vs Next-token training

Run this cell independently once results exist. Overlay Stage 1a (purple) and Stage 3 (green); solid circles show next-token training and dashed squares show last-token training. Both use the same last-token probe task. The x-axis is k/D, D=12; the y-axis is normalized CE on a log scale. Only the first and last theoretical plateaus are displayed.

| Stage | Next-token checkpoint | Last-token checkpoint |
|---|---:|---:|
| 1a | 4096 | 512 |
| 3 | 1048576 | 2097152 |


In [ ]:
# Reload the current plotting script on every run.
import runpy
comparison_plots = runpy.run_path('plot_last_token_vs_next_token_training.py')
